# NHL-Beyond-27 · Book 3 — Spicy & Spicy-Weighted Analysis

Repo: `ewnike/NHL-Beyond-27` — *MADS Milestone I Project*  
Python: 3.13.7 (pyenv env: `nhl_beyond27-3.13.7`)  
Editors/Tools: VSCode, Git/GitHub, Postgres + pgAdmin  
Logs: written via `log_utils.py` (default `logs/`)

**This notebook covers:**
1) Load analysis view/table  
2) Define roles (Defense vs Forwards)  
3) Averages of **spicy** and **spicy_weighted** by role over `rel_age ∈ {-2,-1,0,1,2}`  
4) Visuals: lines with error bars, histograms, and scatter (spicy vs spicy_weighted)  
5) Optional: per-player 5-year averages table for quick reference


**Run this to set paths, helpers...**

In [1]:
# UNIVERSAL BOOTSTRAP
import platform
import re
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

CWD = Path.cwd()
ROOT = CWD if (CWD / "src").exists() and (CWD / "data").exists() else CWD.parent
assert (ROOT / "src").exists() and (ROOT / "data").exists(), "Run from project root or notebooks/."

if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))

ASSETS_DIR  = ROOT / "data" / "assets"
OUTPUTS_DIR = ROOT / "data" / "outputs"
FIGS_DIR    = OUTPUTS_DIR / "figs"
ASSETS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
FIGS_DIR.mkdir(parents=True, exist_ok=True)

def _safe(s):
    import re; return re.sub(r"[^A-Za-z0-9_\-]+","_",str(s)).strip("_")  # noqa: E702

def list_assets():
    return sorted({p.stem for p in ASSETS_DIR.glob("*.csv")} | {p.stem for p in ASSETS_DIR.glob("*.csv.gz")})

def load_asset(stem: str):
    for ext in (".csv",".csv.gz"):
        p = ASSETS_DIR / f"{stem}{ext}"
        if p.exists():
            df = pd.read_csv(p, compression="infer")
            if "rel_age" in df.columns:
                df["rel_age"] = pd.to_numeric(df["rel_age"], errors="coerce")
            return df
    raise FileNotFoundError(f"Missing {stem}.csv in {ASSETS_DIR}")

def require_cols(df, cols, name="df"):
    miss = set(cols) - set(df.columns)
    assert not miss, f"{name} missing columns: {sorted(miss)}"

def save_output_csv(df, name, *, versioned=False, index=False, float_format=None):
    from datetime import datetime, timezone
    stem = _safe(name) + (f"-{datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S')}" if versioned else "")
    path = OUTPUTS_DIR / f"{stem}.csv"
    df.to_csv(path, index=index, float_format=float_format); print(f"[outputs] {path.name}  rows={len(df):,}  cols={df.shape[1]}")
    return str(path)

def save_fig_matplotlib(fig, name, dpi=180):
    path = FIGS_DIR / f"{_safe(name)}.png"; fig.savefig(path, dpi=dpi, bbox_inches="tight"); print(f"[fig] {path.name}"); return str(path)

print(f"Python {platform.python_version()}  |  Pandas {pd.__version__}  |  Numpy {np.__version__}")
print("Assets:", ASSETS_DIR.resolve()); print("Outputs:", OUTPUTS_DIR.resolve())
print("Available asset tables:", list_assets())


Python 3.13.7  |  Pandas 2.3.2  |  Numpy 2.3.3
Assets: /Users/ericwiniecke/Documents/github/NHL-Beyond-27/data/assets
Outputs: /Users/ericwiniecke/Documents/github/NHL-Beyond-27/data/outputs
Available asset tables: ['peak_player_season_stats', 'player_five_year_aligned', 'player_five_year_aligned_cap', 'player_five_year_aligned_z', 'player_five_year_aligned_z_cohort', 'player_hockeyref_even_toi', 'player_hockeyref_even_toi_sample_200', 'player_peak_season', 'player_peak_season_one_row', 'player_streak_seasons']


**Load Necessary Assets**

In [2]:
# === Load the correct assets ===
df_z        = load_asset("player_five_year_aligned_z")
df_z_cohort = load_asset("player_five_year_aligned_z_cohort")

# --- Normalize column names we rely on
for df in (df_z, df_z_cohort):
    # sometimes 'pos' instead of 'position'
    if "position" not in df.columns and "pos" in df.columns:
        df.rename(columns={"pos": "position"}, inplace=True)

    # derive role if missing (D vs F) from position
    if "role" not in df.columns:
        if "position" in df.columns:
            pos0 = df["position"].astype(str).str.upper().str[0]
            df["role"] = np.where(pos0.eq("D"), "D", "F")
        else:
            raise KeyError("Neither 'role' nor 'position' exists to infer role.")

# --- Required columns for downstream
need = {"player","position","role","season","peak_year","rel_age"}
require_cols(df_z,        need, "df_z")
require_cols(df_z_cohort, need, "df_z_cohort")

# --- Ensure rel_age is numeric and in expected set
VALID_RELS = {-2, -1, 0, 1, 2}
df_z["rel_age"]        = pd.to_numeric(df_z["rel_age"], errors="coerce")
df_z_cohort["rel_age"] = pd.to_numeric(df_z_cohort["rel_age"], errors="coerce")

print("[ok] core assets loaded")
print("df_z    shape:", df_z.shape,    "| roles:", df_z["role"].value_counts(dropna=False).to_dict())
print("df_coh  shape:", df_z_cohort.shape, "| roles:", df_z_cohort["role"].value_counts(dropna=False).to_dict())


[ok] core assets loaded
df_z    shape: (1410, 22) | roles: {'F': 900, 'D': 510}
df_coh  shape: (1410, 16) | roles: {'F': 900, 'D': 510}


**Cache canonical copies to data/outputs**

In [3]:
# Cache canonical copies (so downstream code can rely on short names)
# Overwrites on each run; doesn't touch data/assets/*
save_output_csv(df_z,        "df_z",        versioned=True)
save_output_csv(df_z_cohort, "df_z_cohort", versioned=True)


[outputs] df_z-20251003-210610.csv  rows=1,410  cols=22
[outputs] df_z_cohort-20251003-210610.csv  rows=1,410  cols=16


'/Users/ericwiniecke/Documents/github/NHL-Beyond-27/data/outputs/df_z_cohort-20251003-210610.csv'

**Smoke Test**

In [4]:
def smoke(df, name): print(f"[smoke] {name}: rows={len(df):,}, cols={df.shape[1]}")
smoke(df_z, "df_z"); smoke(df_z_cohort, "df_z_cohort"); print("[ok] Preflight checks passed")


[smoke] df_z: rows=1,410, cols=22
[smoke] df_z_cohort: rows=1,410, cols=16
[ok] Preflight checks passed


**Imports**

In [8]:
# Minimal viz imports (put near the top of the notebook)
# Verify imports
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import statsmodels.api as sm
import statsmodels.formula.api as smf

print("numpy", np.__version__)
print("pandas", pd.__version__)
print("statsmodels", sm.__version__)



numpy 2.3.3
pandas 2.3.2
statsmodels 0.14.5


In [ ]:
# from pathlib import Path

# import pandas as pd

# USE_DB = True  # set False to force CSV path

# df = None
# err = None

# if USE_DB:
#     try:
#         from db_utils import get_db_engine

#         engine = get_db_engine()
#         # View columns expected:
#         # player, position, peak_year, rel_age, season, age,
#         # cf_pct_z, cf60_z, ca60_z, spicy_score, spicy_weighted,
#         # cf_pct_dz, cf60_dz, ca60_dz, spicy_unw_dz, spicy_w_dz
#         df = pd.read_sql("SELECT * FROM public.v_player_spicy_by_rel_age", engine)
#         print(f"Loaded {len(df):,} rows from DB view.")
#     except Exception as e:
#         err = e
#         print("DB load failed; will try CSV fallback. Details:", e)

# if df is None:
#     # Fallback CSV path — adjust if you exported under a different name
#     csv_fallback = Path("data/outputs/player_five_year_aligned_z.csv")
#     assert (
#         csv_fallback.exists()
#     ), "CSV fallback not found. Export the z-table to CSV or enable DB connection."
#     df = pd.read_csv(csv_fallback)
#     print(f"Loaded {len(df):,} rows from CSV fallback:", csv_fallback)

# df.head(5)

In [ ]:
# from pathlib import Path

# import pandas as pd

# # Where we'll store exports so future notebook runs don't need DB access
# OUT = Path("data/outputs")
# OUT.mkdir(parents=True, exist_ok=True)

# df = None
# try:
#     from db_utils import get_db_engine

#     eng = get_db_engine()

#     # Try the view first
#     try:
#         df = pd.read_sql("SELECT * FROM public.v_player_spicy_by_rel_age", eng)
#         print("Loaded view: public.v_player_spicy_by_rel_age", df.shape)
#         df.to_csv(OUT / "v_player_spicy_by_rel_age.csv", index=False)
#         print("Exported CSV:", OUT / "v_player_spicy_by_rel_age.csv")
#     except Exception:
#         # Fallback: base z-table
#         df = pd.read_sql("SELECT * FROM public.player_five_year_aligned_z", eng)
#         print("Loaded table: public.player_five_year_aligned_z", df.shape)
#         df.to_csv(OUT / "player_five_year_aligned_z.csv", index=False)
#         print("Exported CSV:", OUT / "player_five_year_aligned_z.csv")

# except Exception as e_db:
#     print("DB load failed (no engine or table/view missing):", e_db)

# if df is None:
#     raise RuntimeError(
#         "Could not load from DB. Either configure DATABASE_URL and build the tables, "
#         "or use Option B below to load from a CSV you’ve already exported."
#     )

# # Keep only columns we need downstream; don’t fail if extras exist
# need = ["player", "position", "rel_age", "spicy_score", "spicy_weighted"]
# have = [c for c in need if c in df.columns]
# print("Columns available for analysis:", have)

In [24]:
import numpy as np
import pandas as pd

# ---- 1) Clean + guard basics
# df = df.copy()
# df["rel_age"] = pd.to_numeric(df.get("rel_age"), errors="coerce")
# df = df.replace([np.inf, -np.inf], np.nan)
# df = df[df["rel_age"].isin([-2, -1, 0, 1, 2])]
# # before the summarize cell
df = df.copy()
if "z_comp_unweighted" not in df and "spicy_unw_dz" in df:
    df["z_comp_unweighted"] = df["spicy_unw_dz"]
if "z_comp_weighted" not in df and "spicy_w_dz" in df:
    df["z_comp_weighted"] = df["spicy_w_dz"]


if "role" not in df.columns:
    if "position" in df.columns:
        df["role"] = np.where(df["position"].astype(str).str.upper().str.startswith("D"), "D", "F")
    else:
        raise KeyError("Need 'role' or 'position' to derive role.")

# ---- 2) Column auto-picker
def pick_col(frame: pd.DataFrame, candidates: list[str], friendly: str) -> str | None:
    for c in candidates:
        if c in frame.columns:
            return c
    return None

unw_candidates = [
    "z_comp_unweighted", "spicy_unw_dz", "spicy_score", "comp_unweighted", "z_unw"
]
w_candidates = [
    "z_comp_weighted", "spicy_w_dz", "spicy_weighted", "comp_weighted", "z_w"
]

col_unw = pick_col(df, unw_candidates, "unweighted")
col_w   = pick_col(df, w_candidates,   "weighted")

print("Detected columns:")
print("  unweighted ->", col_unw if col_unw else "(not found)")
print("  weighted   ->", col_w   if col_w   else "(not found)")

if not col_unw and not col_w:
    raise KeyError(
        "Could not find any composite columns. "
        f"Tried unweighted={unw_candidates} and weighted={w_candidates}. "
        f"Available: {sorted(df.columns)}"
    )

# ---- 3) Summarizer
def summarize_ci(frame: pd.DataFrame, col: str) -> pd.DataFrame:
    s = pd.to_numeric(frame[col], errors="coerce")
    g = (
        frame.assign(_val=s)
        .groupby(["role", "rel_age"], observed=True)["_val"]
        .agg(n="count", mean="mean", std=lambda x: x.std(ddof=1))
        .reset_index()
    )
    g["se"] = g["std"] / np.sqrt(g["n"].clip(lower=1))
    g["ci95"] = np.where(g["n"] > 1, 1.96 * g["se"], np.nan)
    g["metric"] = col
    return g

# ---- 4) Compute summaries (only for columns that exist)
summaries = []
if col_unw:
    agg_unw = summarize_ci(df, col_unw)
    summaries.append(agg_unw)
    display(agg_unw.head(10).style.set_caption(f"Summary (unweighted: {col_unw})"))
if col_w:
    agg_w = summarize_ci(df, col_w)
    summaries.append(agg_w)
    display(agg_w.head(10).style.set_caption(f"Summary (weighted: {col_w})"))

# Optional: combined table
if summaries:
    agg_both = pd.concat(summaries, ignore_index=True)
    print(f"\nCombined summary rows: {len(agg_both)}")


Detected columns:
  unweighted -> spicy_score
  weighted   -> (not found)


,role,rel_age,n,mean,std,se,ci95,metric
0,D,-2,102,-0.015339,0.856616,0.084818,0.166243,spicy_score
1,D,-1,102,-0.015424,0.865200,0.085668,0.167908,spicy_score
2,D,0,102,-0.000329,0.824800,0.081667,0.160068,spicy_score
3,D,1,102,-0.052849,0.909307,0.090035,0.176468,spicy_score
4,D,2,102,0.002871,0.779021,0.077135,0.151184,spicy_score
5,F,-2,180,0.008692,0.882587,0.065784,0.128937,spicy_score
6,F,-1,180,0.008740,0.877863,0.065432,0.128247,spicy_score
7,F,0,180,0.000186,0.892089,0.066492,0.130325,spicy_score
8,F,1,180,0.029948,0.848787,0.063265,0.123999,spicy_score
9,F,2,180,-0.001627,0.932277,0.069488,0.136196,spicy_score



Combined summary rows: 10


In [25]:
import plotly.express

fig1 = px.line(
    agg_spicy,
    x="rel_age",
    y="mean",
    color="role",
    error_y="ci95",
    markers=True,
    title="Mean Spicy (±95% CI) by rel_age and Role",
    labels={"mean": "spicy (z)", "rel_age": "rel_age", "role": "Role"},
)
fig1.update_layout(yaxis=dict(zeroline=True))  # noqa: C408
fig1.show()

fig2 = px.line(
    agg_spicy_w,
    x="rel_age",
    y="mean",
    color="role",
    error_y="ci95",
    markers=True,
    title="Mean Spicy-Weighted (±95% CI) by rel_age and Role",
    labels={"mean": "spicy_weighted (z)", "rel_age": "rel_age", "role": "Role"},
)
fig2.update_layout(yaxis=dict(zeroline=True))  # noqa: C408
fig2.show()

NameError: name 'agg_spicy_w' is not defined

In [ ]:
fig_h = px.histogram(
    df,
    x="spicy_weighted",
    color="role",
    barmode="overlay",
    nbins=40,
    title="Distribution of spicy_weighted by Role (all rel_age pooled)",
    labels={"spicy_weighted": "spicy_weighted (z)"},
    opacity=0.65,
)
fig_h.update_layout(bargap=0.02)
fig_h.show()

In [26]:
order = [-2, -1, 0, 1, 2]
df["rel_age"] = pd.Categorical(
    pd.to_numeric(df["rel_age"], errors="coerce"), categories=order, ordered=True
)


fig_sc = px.scatter(
    df,
    x="spicy_score",
    y="spicy_weighted",
    color="role",
    symbol="role",
    facet_col="rel_age",
    facet_col_wrap=5,
    trendline="ols",  # if statsmodels is available; otherwise remove this line
    title="Spicy vs Spicy-Weighted by Role (faceted by rel_age)",
    labels={"spicy_score": "spicy", "spicy_weighted": "spicy_weighted"},
    opacity=0.6,
    category_orders={"rel_age": order, "role": ["D", "F"]},
)
fig_sc.update_layout(showlegend=True)
fig_sc.show()

ValueError: Value of 'y' is not the name of a column in 'data_frame'. Expected one of ['player', 'position', 'peak_year', 'rel_age', 'start_year', 'season', 'age', 'cf_pct', 'cf60', 'ca60', 'cf_pct_z', 'cf60_z', 'ca60_z', 'spicy_score', 'created_at', 'role'] but received: spicy_weighted

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px

# Order facets consistently
order = [-2, -1, 0, 1, 2]

df = df.copy()

# Ensure role is just "D" vs "F" for plotting
df["role"] = np.where(
    df["role"].astype(str).str.upper().str.startswith("D"),
    "D",
    "F",
)

# Force categorical ordering for rel_age
df["rel_age"] = pd.Categorical(
    pd.to_numeric(df["rel_age"], errors="coerce"),
    categories=order,
    ordered=True,
)


def annotate_trendline_pvals(fig):
    """
    Adds 'p (slope) = ...' to each OLS trendline in a (possibly faceted) PX figure.
    Requires trendline='ols' and statsmodels installed.
    """
    # One row per trendline (group x facet)
    res_df = px.get_trendline_results(fig)

    # Find the trendline traces (the straight lines)
    trendline_traces = [
        i
        for i, tr in enumerate(fig.data)
        if getattr(tr, "mode", None) == "lines" and "trendline" in tr.name.lower()
    ]

    n = min(len(trendline_traces), len(res_df))
    for k in range(n):
        tr = fig.data[trendline_traces[k]]
        fit = res_df.iloc[k]["px_fit_results"]  # statsmodels OLSResults

        # Prefer the slope term (not Intercept/const)
        names = list(fit.pvalues.index)
        slope_name = next(
            (nm for nm in names if nm.lower() not in ("intercept", "const")), names[-1]
        )
        p_val = float(fit.pvalues[slope_name])

        # Annotate inside that subplot’s domain (works with facets)
        xref = f"{tr.xaxis} domain"
        yref = f"{tr.yaxis} domain"
        fig.add_annotation(
            x=0.03,
            y=0.95,
            xref=xref,
            yref=yref,
            text=f"p (slope) = {p_val:.3g}",
            showarrow=False,
            font=dict(size=10),
            bgcolor="rgba(255,255,255,0.6)",
        )


def make_scatter(sub: pd.DataFrame, role_label: str):
    return px.scatter(
        sub,
        x="spicy_score",
        y="spicy_weighted",
        color="rel_age",  # color by rel_age (role is constant in each figure)
        symbol="rel_age",
        facet_col="rel_age",
        facet_col_wrap=5,
        trendline="ols",  # remove if statsmodels isn't available
        title=f"Spicy vs Spicy-Weighted — {role_label}",
        labels={
            "spicy_score": "spicy",
            "spicy_weighted": "spicy_weighted",
            "rel_age": "rel_age",
        },
        opacity=0.6,
        category_orders={"rel_age": order},
    )


# 1) Defense
fig_d = make_scatter(df[df["role"] == "D"], "Defense (D)")
annotate_trendline_pvals(fig_d)
fig_d.update_layout(showlegend=True)
fig_d.show()

# 2) Forwards
fig_f = make_scatter(df[df["role"] == "F"], "Forwards (F)")
annotate_trendline_pvals(fig_f)
fig_f.update_layout(showlegend=True)
fig_f.show()

In [ ]:
summary = (
    df.groupby(["role", "rel_age"])
    .agg(
        spicy_mean=("spicy_score", "mean"),
        spicy_w_mean=("spicy_weighted", "mean"),
        n=("spicy_score", "count"),
    )
    .reset_index()
    .sort_values(["role", "rel_age"])
)
summary.round({"spicy_mean": 3, "spicy_w_mean": 3}).head(20)

In [ ]:
player_5yr = (
    df.groupby(["player", "role"])
    .agg(
        spicy_mean=("spicy_score", "mean"),
        spicy_std=("spicy_score", "std"),
        spicy_w_mean=("spicy_weighted", "mean"),
        spicy_w_std=("spicy_weighted", "std"),
        years=("rel_age", "nunique"),
    )
    .reset_index()
)
player_5yr = player_5yr[
    player_5yr["years"].ge(3)
]  # keep players with ≥3 seasons in window (optional)
player_5yr.round(3).head(20)

In [ ]:
OUT = Path("data/outputs")
OUT.mkdir(parents=True, exist_ok=True)

summary.to_csv(OUT / "spicy_summary_by_role_rel_age.csv", index=False)
player_5yr.to_csv(OUT / "spicy_player_5yr_summary.csv", index=False)

print(
    "Wrote:", OUT / "spicy_summary_by_role_rel_age.csv", "and", OUT / "spicy_player_5yr_summary.csv"
)

## Notes & interpretation

- **Within-player standardization**: both spicy and spicy_weighted are in z-score units relative to each player’s own 5-year baseline.
- **Role split**: we classify `position` starting with `D` as Defense, everything else as Forwards (C/LW/RW…).
- **rel_age axis**: `-2,-1,0,1,2` aligns to ages like 25–29 in our cohort build (peak = 0).
- **What to look for**:
  - Do Defense (“D”) show flatter/less volatile spicy_weighted curves than Forwards?
  - Does the weighting visibly nudge defensemen down (penalizing CA/60) and forwards up (rewarding CF/60)?
  - Are histograms tighter/wider by role?


In [ ]:
wide = summary.pivot(index="rel_age", columns="role", values="spicy_w_mean")
wide["F_minus_D"] = wide.get("F", pd.Series(index=wide.index)) - wide.get(
    "D", pd.Series(index=wide.index)
)
wide.round(3)

In [ ]:
import numpy as np
import pandas as pd

# statsmodels for OLS
import statsmodels.formula.api as smf  # noqa: F811

# We’ll reuse df from earlier cells. Ensure it has the columns we need and no infs/NAs in the target.
need = {"player", "role", "rel_age", "spicy_score", "spicy_weighted"}
missing = need - set(df.columns)
if missing:
    raise ValueError(f"Missing columns for regression: {sorted(missing)}")

reg = df.copy()
reg = reg.replace([np.inf, -np.inf], np.nan)

# Keep the rel_age window we care about and drop rows missing the target
reg = reg[reg["rel_age"].isin([-2, -1, 0, 1, 2])].copy()


# Role: ensure exactly {"D","F"} with "D" as reference in the model
def to_role(pos):
    s = str(pos or "").strip().upper()
    return "D" if s.startswith("D") else "F"


reg["role"] = reg["role"].map(lambda x: "D" if str(x).upper().startswith("D") else "F")

# Drop rows without outcome
reg_sw = reg.dropna(subset=["spicy_weighted"]).copy()
reg_s = reg.dropna(subset=["spicy_score"]).copy()

print(f"Rows for spicy_weighted: {len(reg_sw):,} | Rows for spicy: {len(reg_s):,}")
reg_sw[["player", "role", "rel_age", "spicy_weighted"]].head(3)

In [ ]:
# C(role, Treatment('D')) makes "D" the reference; C(rel_age, Treatment(0)) makes 0 the reference
fml_sw = "spicy_weighted ~ C(role, Treatment('D')) * C(rel_age, Treatment(0))"

model_sw = smf.ols(formula=fml_sw, data=reg_sw).fit(cov_type="HC3")
print(model_sw.summary())

In [ ]:
fml_s = "spicy_score ~ C(role, Treatment('D')) * C(rel_age, Treatment(0))"

model_s = smf.ols(formula=fml_s, data=reg_s).fit(cov_type="HC3")
print(model_s.summary())

In [ ]:
# Cluster-robust by player; useful if multiple rows per player (which we have)
# Note: statsmodels accepts group labels as strings.
model_sw_cl = smf.ols(formula=fml_sw, data=reg_sw).fit(
    cov_type="cluster",
    cov_kwds={"groups": reg_sw["player"]},
)
print(model_sw_cl.summary())

In [ ]:
import plotly.express as px

# Grid of unique combos to predict
grid = pd.MultiIndex.from_product(
    [["D", "F"], [-2, -1, 0, 1, 2]], names=["role", "rel_age"]
).to_frame(index=False)

# Predict with CI from the HC3 model
pred = model_sw.get_prediction(grid).summary_frame(alpha=0.05)
pred = pd.concat([grid.reset_index(drop=True), pred.reset_index(drop=True)], axis=1)
pred.rename(
    columns={"mean": "yhat", "mean_ci_lower": "ci_lo", "mean_ci_upper": "ci_hi"}, inplace=True
)

pred.head(10)

In [ ]:
fig = go.Figure()

for role, g in pred.groupby("role"):
    g = g.sort_values("rel_age")
    # CI ribbon
    fig.add_traces(
        [
            go.Scatter(
                x=g["rel_age"],
                y=g["ci_hi"],
                mode="lines",
                line=dict(width=0),
                showlegend=False,
                hoverinfo="skip",
            ),
            go.Scatter(
                x=g["rel_age"],
                y=g["ci_lo"],
                mode="lines",
                line=dict(width=0),
                fill="tonexty",
                name=f"{role} 95% CI",
                hoverinfo="skip",
                opacity=0.2,
            ),
        ]
    )
    # Mean line
    fig.add_trace(
        go.Scatter(
            x=g["rel_age"],
            y=g["yhat"],
            mode="lines+markers",
            name=f"{role} mean",
        )
    )

fig.update_layout(
    title="Predicted spicy_weighted by rel_age × role (OLS, HC3 CIs)",
    xaxis_title="rel_age",
    yaxis_title="spicy_weighted (z)",
    legend_title="Series",
)
fig.show()

In [ ]:
import pandas as pd

# Build a tidy coef table directly from the model object (version-agnostic)
coefs = (
    pd.concat(
        [
            model_sw.params.rename("coef"),
            model_sw.bse.rename("se"),
            model_sw.pvalues.rename("p"),
        ],
        axis=1,
    )
    .reset_index()
    .rename(columns={"index": "term"})
)

# Example: p-value for the Forward-vs-Defense contrast at rel_age=0
coefs.loc[coefs["term"] == "C(role, Treatment('D'))[T.F]", "p"]

# Helpful, human-readable labels (keep flexible—these term strings come from patsy)
coefs["label"] = coefs["term"].replace(
    {
        "Intercept": "D @ rel_age=0",
        "C(role, Treatment('D'))[T.F]": "F vs D (at rel_age=0)",
        "C(rel_age, Treatment(0))[T.-2]": "rel_age = -2 (D baseline)",
        "C(rel_age, Treatment(0))[T.-1]": "rel_age = -1 (D baseline)",
        "C(rel_age, Treatment(0))[T.1]": "rel_age = 1 (D baseline)",
        "C(rel_age, Treatment(0))[T.2]": "rel_age = 2 (D baseline)",
        "C(role, Treatment('D'))[T.F]:C(rel_age, Treatment(0))[T.-2]": "Interaction F×(-2)",
        "C(role, Treatment('D'))[T.F]:C(rel_age, Treatment(0))[T.-1]": "Interaction F×(-1)",
        "C(role, Treatment('D'))[T.F]:C(rel_age, Treatment(0))[T.1]": "Interaction F×1",
        "C(role, Treatment('D'))[T.F]:C(rel_age, Treatment(0))[T.2]": "Interaction F×2",
    }
)

coefs.head()

In [ ]:
# --- Clean version: NO RIBBONS, just lines + asymmetric CI bars ---

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import statsmodels.formula.api as smf

# Expect df with: player, role/position, rel_age, spicy_weighted, (optional) spicy_score
df0 = df.copy()
role_col = "position" if "position" in df0.columns else "role"
df0["role"] = df0[role_col].astype(str).str.upper().str.startswith("D").map({True: "D", False: "F"})

order = [-2, -1, 0, 1, 2]
df0["rel_age"] = pd.to_numeric(df0["rel_age"], errors="coerce")
df0 = df0[df0["rel_age"].isin(order)].copy()
df0["rel_age"] = pd.Categorical(df0["rel_age"], categories=order, ordered=True)

# 1) Group means ± SE (lines only; error bars = SE)
g = (
    df0.groupby(["role", "rel_age"], observed=True)["spicy_weighted"]
    .agg(mean="mean", sd="std", n="size")
    .reset_index()
)
g["se"] = g["sd"] / np.sqrt(g["n"].clip(lower=1))

fig_me = px.line(
    g.sort_values(["role", "rel_age"]),
    x="rel_age",
    y="mean",
    color="role",
    markers=True,
    error_y="se",
    category_orders={"rel_age": order, "role": ["D", "F"]},
    title="Mean spicy_weighted ± SE by Role × rel_age (no ribbons)",
    labels={"mean": "mean(spicy_weighted)"},
)
fig_me.update_traces(mode="lines+markers")
fig_me.update_yaxes(tickformat=".2f", dtick=0.05)
fig_me.show()

# 2) Modeled curvature (Δ from peak), NO ribbons — just CI error bars on the modeled mean
#    Build Δz from peak per player, fit quadratic with role interactions
base0 = (
    df0[df0["rel_age"].astype(int) == 0].groupby("player")["spicy_weighted"].first().rename("base0")
)
df2 = df0.join(base0, on="player", how="left")
df2 = df2.dropna(subset=["base0"]).copy()
df2["spicy_w_dz"] = df2["spicy_weighted"] - df2["base0"]
df2["rel_age_num"] = df2["rel_age"].astype(int)

m = smf.ols("spicy_w_dz ~ role * rel_age_num + role * I(rel_age_num**2)", data=df2).fit(
    cov_type="HC3"
)

grid = pd.MultiIndex.from_product([["D", "F"], order], names=["role", "rel_age_num"]).to_frame(
    index=False
)
pred = m.get_prediction(grid).summary_frame(alpha=0.05)
grid["yhat"] = pred["mean"].values
grid["err_plus"] = (pred["mean_ci_upper"] - pred["mean"]).values
grid["err_minus"] = (pred["mean"] - pred["mean_ci_lower"]).values
grid["rel_age"] = pd.Categorical(grid["rel_age_num"], categories=order, ordered=True)

fig_m = go.Figure()
palette = {"D": "#1f77b4", "F": "#ff7f0e"}
for r in ["D", "F"]:
    sub = grid[grid["role"] == r].sort_values("rel_age_num")
    fig_m.add_trace(
        go.Scatter(
            x=sub["rel_age"],
            y=sub["yhat"],
            mode="lines+markers",
            name=f"{r} mean (modeled)",
            line=dict(color=palette.get(r, "#888"), width=2),
            marker=dict(size=7),
            error_y=dict(array=sub["err_plus"], arrayminus=sub["err_minus"], visible=True),
        )
    )

fig_m.update_layout(
    title="Modeled curvature of spicy_weighted (Δ from peak) by Role — no ribbons",
    xaxis_title="rel_age",
    yaxis_title="Δ spicy_weighted (vs player peak at rel_age=0)",
    template="plotly_white",
    legend=dict(title=""),
)
fig_m.update_xaxes(tickvals=order)
fig_m.update_yaxes(tickformat=".2f", dtick=0.05, zeroline=True)
fig_m.show()

# 3) Peak Δ teaser: compare role-weighted vs equal-weight (if spicy_score available)
#    Peak Δ = z_0 - mean(z_t) over the window = - mean(Δz)
out_msgs = []
if "spicy_score" in df0.columns:
    # compute Peak Δ for both composites
    win = df0.copy()
    # role-weighted Peak Δ
    peak_w = win.groupby("player")["spicy_w_dz"].mean().mul(-1).rename("peak_delta_w")
    # equal-weight Δz on the fly from levels (if you have spicy_score at t)
    base_eq = (
        win[win["rel_age"].astype(int) == 0]
        .groupby("player")["spicy_score"]
        .first()
        .rename("base0_eq")
    )
    tmp = win.join(base_eq, on="player", how="left")
    tmp = tmp.dropna(subset=["base0_eq"])
    tmp["spicy_eq_dz"] = tmp["spicy_score"] - tmp["base0_eq"]
    peak_eq = tmp.groupby("player")["spicy_eq_dz"].mean().mul(-1).rename("peak_delta_eq")

    roles = win.groupby("player")["role"].first()
    peaks = pd.concat([peak_w, peak_eq, roles], axis=1).dropna()

    # scatter with 45° reference
    import plotly.express as px

    fig_pd = px.scatter(
        peaks.reset_index(),
        x="peak_delta_eq",
        y="peak_delta_w",
        color="role",
        opacity=0.65,
        trendline="ols",
        trendline_scope="overall",
        labels={
            "peak_delta_eq": "Peak Δ (SPICY equal-weight)",
            "peak_delta_w": "Peak Δ (SPICY role-weighted)",
        },
        title="Peak Δ: Role-weighted vs Equal-weight (per player)",
        category_orders={"role": ["D", "F"]},
    )
    # add 45-degree line
    mn = float(np.nanmin([peaks["peak_delta_eq"].min(), peaks["peak_delta_w"].min()]))
    mx = float(np.nanmax([peaks["peak_delta_eq"].max(), peaks["peak_delta_w"].max()]))
    pad = (mx - mn) * 0.05 if np.isfinite(mx - mn) else 0.1
    rng = [mn - pad, mx + pad]
    fig_pd.add_trace(
        go.Scatter(
            x=rng, y=rng, mode="lines", name="y=x", line=dict(width=1, dash="dash"), showlegend=True
        )
    )
    fig_pd.update_xaxes(range=rng, tickformat=".2f")
    fig_pd.update_yaxes(range=rng, tickformat=".2f")
    fig_pd.show()

    # quick correlation
    corr = peaks[["peak_delta_eq", "peak_delta_w"]].corr().iloc[0, 1]
    out_msgs.append(f"Peak Δ corr (eq vs weighted): {corr:.3f}")

# 4) Small regression: does weighting change peak height by role?
#    Model Peak Δ_w ~ role (and optionally controls)
if "spicy_w_dz" in df2.columns:
    peak_delta_w = (
        df2.groupby("player")["spicy_w_dz"].mean().mul(-1).rename("peak_delta_w")
    ).to_frame()
    peak_delta_w["role"] = df2.groupby("player")["role"].first()
    m_pd = smf.ols("peak_delta_w ~ role", data=peak_delta_w).fit(cov_type="HC3")
    out_msgs.append("Peak Δ_w by role regression (coef on role[T.F] is F - D):")
    out_msgs.append(str(m_pd.summary().tables[1]))

print("\n".join(out_msgs))

explanation of the above graphs:  Short version: you’re seeing some D points land **above the y=x line** because our role-weighted formula gives **extra credit to suppression (−CA60_z)** for defensemen. If a D’s **suppression is stronger than their creation** at peak, the **weighted** composite will come out a bit higher than the **equal-weighted** one—so those dots sit above the diagonal.

Longer take:

* The scatter is **Peak Δ (role-weighted) on the y-axis vs Peak Δ (equal-weight) on the x-axis**.

  * Points **on** the dashed y=x line → weighting made no difference.
  * Points **above** the line → weighting **increased** the composite for that player (relative to equal-weight).
  * Points **below** → weighting **reduced** it.

* Our role weights:

  * **Defense:** `0.5*cf_pct_z + 0.2*cf60_z – 0.3*ca60_z` (more emphasis on suppression).
  * **Forwards:** `0.5*cf_pct_z + 0.3*cf60_z – 0.2*ca60_z` (more emphasis on creation).

* If a defenseman’s **CA60_z is especially good** (i.e., they suppress shots; *lower* CA60_z → bigger positive contribution after the minus sign), the D-weights boost them a touch versus equal-weight. That’s why you can see some D points trending a hair **above** y=x at peak.

* But the **headline stats** say the average effect is basically zero:

  * **Correlation of peak Δ (weighted vs equal):** 0.996 → almost a line.
  * Simple role dummy regression (F − D at peak): coef ≈ **0.004**, **p ≈ 0.97** → **no meaningful average role shift**.

So: the weighting scheme does what it’s supposed to (reward suppression for D, creation for F), and you see that for **individual** D’s whose strengths skew toward suppression. But **on average** the two composites are nearly identical; weighting mostly **tightens the band** (less variance) rather than moving the center.


## Regression interpretation (quick)

### A) Categorical model
**Formula:** `spicy_weighted ~ C(role) * C(rel_age, Treatment(0))`

- **Intercept** → Defense at `rel_age = 0` (≈ 0 because outcomes are within-player z’s).
- **`C(role)[T.F]`** → Forwards − Defense **at `rel_age = 0`** (peak difference).
- **`C(rel_age)[T.k]`** (k ∈ {−2, −1, 1, 2}) → Defense at k minus Defense at 0 (D’s deviation from peak).
- **`C(role)[T.F] : C(rel_age)[T.k]`** → extra Forward–Defense gap at k beyond the gap at 0.
- **SEs / p-values:** HC3 robust SEs; p-values can be large because the outcome is standardized within player.

**How to read**
- If `C(role)[T.F] ≈ 0` → F and D look similar at peak.
- Negative `C(rel_age)[T.1]` → D drops post-peak; positive → improves.
- Negative interaction at `k = 1` → F declines from peak faster than D at `+1` (positive → the opposite).

---

### B) Delta-from-peak model
**Formula:** `spicy_w_dz ~ role * rel_age_num + role * I(rel_age_num**2)`  
where `spicy_w_dz = spicy_weighted − (player’s value at rel_age = 0)`.

- **Outcome** is centered per player at peak → focuses on **trajectory/curvature**.
- **Intercept** → Defense at `rel_age = 0` (**exactly 0** by construction).
- **`rel_age_num`** → slope per step from peak for Defense.
- **`I(rel_age_num**2)`** → curvature for Defense (negative → concave decline away from peak).
- **`role[T.F]`** → F vs D at peak (should be ~0 now).
- **`role[T.F] : rel_age_num`** → slope difference F vs D.
- **`role[T.F] : I(rel_age_num**2)`** → curvature difference F vs D (key for “who ages faster/slower”).

**How to read**
- Quadratic < 0 and significant → performance bends downward away from peak.
- `role[T.F] : I(rel_age_num**2)` < 0 → Forwards show stronger concavity (faster decline) than Defense; > 0 → Defense declines faster.
- Use the **prediction plot with 95% CIs** to present these effects clearly.

---

### General notes
- Facet/ordering choices (e.g., forcing `rel_age` to −2, −1, 0, 1, 2) affect visuals only, **not** estimates.
- Prefer **effect sizes + 95% CIs** over binary significance.
- With repeated measures, use **clustered SEs by player** (coefficients unchanged; SEs/p-values adjust for within-player dependence).


In [ ]:
reg_sw_lin = reg_sw.copy()
# Ensure rel_age is numeric
reg_sw_lin["rel_age"] = reg_sw_lin["rel_age"].astype(int)

fml_sw_lin = "spicy_weighted ~ C(role, Treatment('D')) + rel_age + C(role, Treatment('D')):rel_age"
model_sw_lin = smf.ols(formula=fml_sw_lin, data=reg_sw_lin).fit(cov_type="HC3")
print(model_sw_lin.summary())

In [ ]:
from pathlib import Path

import pandas as pd

OUT = Path("data/outputs")
OUT.mkdir(parents=True, exist_ok=True)

# 1) Predictions by role × rel_age (from HC3 model)
pred_out = pred.copy()
pred_path = OUT / "reg_spicy_weighted_preds_by_role_rel_age.csv"
pred_out.to_csv(pred_path, index=False)

# 2) Coefficients / SEs / p-values (HC3)
effects_out = coefs.copy()  # built earlier from model_sw.params/bse/pvalues
effects_path = OUT / "reg_spicy_weighted_effects_hc3.csv"
effects_out.to_csv(effects_path, index=False)

# 3) Optional: clustered SEs by player (same coefficients, different SE/p)
try:
    # Only run if the clustered model exists in this session
    _ = model_sw_cl  # will NameError if not defined
    coefs_cl = (
        pd.concat(
            [
                model_sw_cl.params.rename("coef"),
                model_sw_cl.bse.rename("se"),
                model_sw_cl.pvalues.rename("p"),
            ],
            axis=1,
        )
        .reset_index()
        .rename(columns={"index": "term"})
    )
    cl_path = OUT / "reg_spicy_weighted_effects_clustered.csv"
    coefs_cl.to_csv(cl_path, index=False)
    print("Wrote clustered SE table:", cl_path)
except NameError:
    print("Clustered SE export skipped: model_sw_cl was not fit in this run.")
except Exception as e:
    print("Clustered SE export skipped (unexpected error):", e)

print("Wrote:", pred_path, "and", effects_path)

In [ ]:
# Save the pretty-printed statsmodels summaries as .txt for appendix/attachments
try:
    (OUT / "reg_spicy_weighted_ols_hc3.txt").write_text(str(model_sw.summary()))
    print("Wrote:", OUT / "reg_spicy_weighted_ols_hc3.txt")
except Exception as e:
    print("Could not write HC3 summary:", e)

try:
    (OUT / "reg_spicy_weighted_ols_clustered.txt").write_text(str(model_sw_cl.summary()))
    print("Wrote:", OUT / "reg_spicy_weighted_ols_clustered.txt")
except Exception as e:
    print("Clustered summary not written (optional):", e)

In [ ]:
# --- Load z-table into df_z (DB first, else CSV fallback) ---
from pathlib import Path

import pandas as pd


def load_z_table():
    # Try Postgres via your db_utils first
    try:
        from db_utils import get_db_engine

        eng = get_db_engine()
        df = pd.read_sql("SELECT * FROM public.player_five_year_aligned_z", eng)
        print(f"Loaded z-table from DB: {len(df):,} rows")
        return df
    except Exception as e:
        print("[Info] DB not available or table missing; using CSV fallback:", e)
        csv = Path("data/outputs/player_five_year_aligned_z.csv")
        assert csv.exists(), "CSV fallback not found: data/outputs/player_five_year_aligned_z.csv"
        df = pd.read_csv(csv)
        print(f"Loaded z-table from CSV: {len(df):,} rows")
        return df


df_z = load_z_table()

# Standardize rel_age ordering
order = [-2, -1, 0, 1, 2]
df_z["rel_age"] = pd.Categorical(
    pd.to_numeric(df_z["rel_age"], errors="coerce"), categories=order, ordered=True
)

# Quick sanity: needed columns
need = {"player", "peak_year", "rel_age", "spicy_score", "spicy_weighted"}
missing = need - set(df_z.columns)
if missing:
    raise KeyError(f"Missing columns in df_z: {sorted(missing)}")

print("df_z ready.")

In [ ]:
# --- Prep: make dfz from whatever you loaded earlier ---
# If you already have df_z from your loader cell:
try:
    dfz = df_z.copy()
except NameError:
    # Fallback: if you used a different name, assign it here
    # dfz = <your_dataframe>.copy()
    raise NameError("df_z (the z-table DataFrame) is not defined. Run the load cell first.")

# Ensure required columns exist
required_cols = {"player", "peak_year", "rel_age", "spicy_score", "spicy_weighted"}
missing = required_cols - set(dfz.columns)
if missing:
    raise KeyError(f"Missing columns in dfz: {sorted(missing)}")

# Make rel_age numeric/categorical in the expected order
order = [-2, -1, 0, 1, 2]
dfz["rel_age"] = pd.Categorical(
    pd.to_numeric(dfz["rel_age"], errors="coerce"), categories=order, ordered=True
)

# (Optional) sanity: drop rows with missing spicy metrics
dfz = dfz.dropna(subset=["spicy_score", "spicy_weighted"])
print(f"dfz ready: {len(dfz):,} rows, columns OK.")

In [ ]:
import numpy as np
import pandas as pd

# Choose which signal to analyze:
Y_COL = "spicy_weighted"  # or "spicy_score"
EPS = 0.05  # tolerance to ignore tiny wiggles
VALID_RELS = [-2, -1, 0, 1, 2]

# Keep only complete 5-point windows
g = (
    dfz.loc[dfz["rel_age"].isin(VALID_RELS), ["player", "peak_year", "rel_age", Y_COL]]
    .dropna()
    .pivot_table(index=["player", "peak_year"], columns="rel_age", values=Y_COL)
)

# ensure all columns present
if not set(VALID_RELS).issubset(g.columns):
    missing = sorted(set(VALID_RELS) - set(g.columns))
    raise ValueError(f"Missing rel_age columns in pivot: {missing}")

g = g[VALID_RELS].dropna()  # require full 5-year window


def is_cap_shaped(vals, r, eps=0.05):
    """vals is array of length 5 for rel_age [-2,-1,0,1,2]. r is -1,0,1."""
    rel = [-2, -1, 0, 1, 2]
    idx = rel.index(r)
    v = vals
    # local peak at r with margin
    if not (v[idx] >= v[idx - 1] + eps and v[idx] >= v[idx + 1] + eps):
        return False
    # monotone up to the peak
    for i in range(idx):
        if not (v[i] + eps <= v[i + 1]):
            return False
    # monotone down after the peak
    for i in range(idx, len(v) - 1):
        if not (v[i] >= v[i + 1] + eps):
            return False
    return True


arr = g.to_numpy()
mask_m1 = np.apply_along_axis(is_cap_shaped, 1, arr, r=-1, eps=EPS)
mask_0 = np.apply_along_axis(is_cap_shaped, 1, arr, r=0, eps=EPS)
mask_p1 = np.apply_along_axis(is_cap_shaped, 1, arr, r=1, eps=EPS)

# Ensure only one peak is counted if multiple happen to pass (prefer center > -1/+1)
peak_at = np.where(
    mask_0, 0, np.where(mask_m1 & ~mask_0, -1, np.where(mask_p1 & ~(mask_0 | mask_m1), 1, np.nan))
)

counts = pd.Series(peak_at, name="peak_rel_age").value_counts(dropna=False).sort_index()
total = len(g)
summary = pd.DataFrame({"count": counts, "percent": (counts / total * 100).round(1)})
summary.index = summary.index.map(
    {-1: "peak@-1", 0: "peak@0", 1: "peak@+1", np.nan: "not_cap_shaped"}
)
summary.loc["TOTAL"] = [total, 100.0]
summary

In [ ]:
# 1) Means + 95% CI for *raw* within-player z (not delta)
order = [-2, -1, 0, 1, 2]
df = df_z.copy()
df["role"] = df["position"].str.upper().str.startswith("D").map({True: "D", False: "F"})
df["rel_age"] = pd.Categorical(
    pd.to_numeric(df["rel_age"], errors="coerce"), categories=order, ordered=True
)


def mean_ci(df, y):
    g = (
        df.groupby(["role", "rel_age"], observed=True)
        .agg(n=(y, "size"), mean=(y, "mean"), sd=(y, "std"))
        .reset_index()
    )
    g["se"] = g["sd"] / g["n"].clip(lower=1).pow(0.5)
    g["lo"] = g["mean"] - 1.96 * g["se"]
    g["hi"] = g["mean"] + 1.96 * g["se"]
    return g


m_raw = mean_ci(df, "spicy_weighted")  # try spicy or your composite z
m_dz = mean_ci(df.assign(y=df["spicy_w_dz"]), "y")

import plotly.express as px

fig_raw = px.line(
    m_raw,
    x="rel_age",
    y="mean",
    color="role",
    error_y=m_raw["hi"] - m_raw["mean"],
    error_y_minus=m_raw["mean"] - m_raw["lo"],
    title="Raw within-player z by rel_age (mean ±95% CI)",
)
fig_raw.show()

fig_dz = px.line(
    m_dz,
    x="rel_age",
    y="mean",
    color="role",
    error_y=m_dz["hi"] - m_dz["mean"],
    error_y_minus=m_dz["mean"] - m_dz["lo"],
    title="Δ-from-peak (z) by rel_age (mean ±95% CI)",
)
fig_dz.show()

**A quick quadratic test on delta from peak:**

In [ ]:
import statsmodels.formula.api as smf

dfq = df.dropna(subset=["spicy_w_dz"]).copy()
dfq["rel_age_num"] = dfq["rel_age"].astype(int)
dfq["role2"] = dfq["role"]  # make sure it's D/F

model = smf.ols("spicy_w_dz ~ role2 * rel_age_num + role2 * I(rel_age_num**2)", data=dfq).fit(
    cov_type="HC3"
)
print(model.summary().tables[1])  # look at the I(rel_age_num**2) terms

**Interpretation: If the quadratic term is negative for both roles, average trajectories are concave (peak at 0, then decline). If you truly saw an “M” (up-down-up) in the averages, the quadratic wouldn’t capture it—LOWESS or splines would—but that pattern is uncommon once you center at each player’s best season and control for edge Ns.**

# Reading the Regression Table

## Columns
- **coef** — Estimated effect (regression coefficient) for the term.
- **std err** — Standard error of the estimate (uncertainty of `coef`).
- **z / t** — Test statistic (≈ `coef / std err`).  
  *statsmodels reports a Wald **z** under robust/HC covariance; with classical OLS it’s typically a **t**.*
- **P>|z| (or P>|t|)** — Two-sided p-value for testing `coef = 0`. Smaller ⇒ stronger evidence the effect ≠ 0.
- **[0.025, 0.975]** — 95% confidence interval for the coefficient.

---

## Terms (given your model)

**Model (paraphrased)**  


**spicy_w_dz ~ C(role) * rel_age_num + C(role) * I(rel_age_num**2)**

- `C(role)` encodes role as a factor (baseline = **D**).  
- `rel_age_num` is centered on peak (`0 ≡ peak year`).  
- `I(rel_age_num**2)` is the quadratic (curvature) term.

**Interpretation of rows**
- **Intercept** — Expected value for **Defense (D)** at `rel_age_num = 0` (the peak).  
  Because the outcome is a *delta-from-peak* (`dz`), this should be ≈ 0 by construction.
- **C(role)[T.F]** — Difference **Forwards − Defense** at the peak (`rel_age_num = 0`).
- **rel_age_num** — Linear change per 1 step away from peak **for Defense**.
- **C(role)[T.F]:rel_age_num** — How the linear slope differs **F vs D**.
- **I(rel_age_num**²)** — Curvature (concavity/convexity) **for Defense**.
- **C(role)[T.F]:I(rel_age_num**²)** — How curvature differs **F vs D**.

> **Reading signs**
> - Negative slope ⇒ metric declines as you move away from peak.  
> - Negative curvature ⇒ concave down (steeper drop away from peak).  
> - Interaction terms tell you how F deviates from D for the same component.

---

## Quick Takeaways (how to summarize)
- Most p-values large ⇒ **no strong evidence** that slopes/curvature differ from zero or by role (at your current N/spec).
- Signs often match intuition (decline from peak; mild concavity), but **CI are wide**.
- To sharpen inference, consider:
  - Cluster-robust SEs by player (`cov_type="cluster", cov_kwds={"groups": df["player"]}`),
  - More data / stricter inclusion filters,
  - Standardizing `rel_age_num` (optional; it can help with scaling),
  - Mixed-effects (random intercepts/slopes by player) if repeated measures are key.

---

## Mini Legend (reporting)
- **Effect size:** report `coef` and **95% CI**: e.g., `β = −0.047 [−0.12, 0.03]`.  
- **Evidence:** report p-value: e.g., `p = 0.21` (not statistically significant).  
- **Practical:** tie back to units (here, **z-units** of the composite).
